# Langfuse v3 — Dataset & Run Explorer / Exporter

Explore datasets, dataset runs, run items (with linked traces + reasoning), then export to **Excel** and **JSON**.

Built for **langfuse v3.x OSS, self-hosted**.

**Workflow:**
1. Configure credentials (cell below)
2. List datasets → pick one
3. Explore: items only, *or* runs + outputs + trace reasoning
4. Export selection to Excel + JSON

> The `langfuse.api.*` namespace is the auto-generated Public-API client. v3 removed the v2 `get_dataset_runs` helper, so we use `api.datasets.*`, `api.dataset_run_items.*`, `api.observations.*` directly.

## 0. Install (run once)

In [ ]:
# %pip install "langfuse>=3,<4" pandas openpyxl --quiet

## 1. Credentials & client

Hardcoded for a private server. **Do not commit this notebook with real keys.**

In [ ]:
from langfuse import Langfuse

LANGFUSE_HOST       = "https://your-langfuse.internal"   # self-hosted URL, no trailing slash
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxx"
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxx"

# Optional: if your org has multiple projects, the keys already scope to one project.
# Verify the project below after auth_check().

langfuse = Langfuse(
    host=LANGFUSE_HOST,
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
)

assert langfuse.auth_check(), "Auth failed — check host/keys"
print("Authenticated OK →", LANGFUSE_HOST)

## 2. Imports & helpers

In [ ]:
import json, datetime as dt
from pathlib import Path
from urllib.parse import quote
import pandas as pd

OUT_DIR = Path("langfuse_exports")
OUT_DIR.mkdir(exist_ok=True)

def _ts():
    return dt.datetime.now().strftime("%Y%m%d_%H%M%S")

def _to_jsonable(obj):
    """Recursively turn pydantic models / objects into plain JSON-serializable structures."""
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {k: _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if hasattr(obj, "model_dump"):          # pydantic v2
        return _to_jsonable(obj.model_dump())
    if hasattr(obj, "dict"):                # pydantic v1
        return _to_jsonable(obj.dict())
    if hasattr(obj, "isoformat"):           # datetimes
        return obj.isoformat()
    return str(obj)

def _flatten(value, max_len=None):
    """Make a cell Excel-safe: dicts/lists -> compact JSON string."""
    if value is None:
        return None
    if isinstance(value, (dict, list)):
        value = json.dumps(_to_jsonable(value), ensure_ascii=False)
    value = str(value)
    if max_len and len(value) > max_len:
        value = value[:max_len] + " …[truncated]"
    return value

def _paginate(list_fn, **kwargs):
    """Generic pager for langfuse.api list endpoints that return .data + .meta."""
    page, out = 1, []
    while True:
        resp = list_fn(page=page, limit=100, **kwargs)
        data = getattr(resp, "data", []) or []
        if not data:
            break
        out.extend(data)
        meta = getattr(resp, "meta", None)
        total_pages = getattr(meta, "total_pages", None) if meta else None
        if total_pages is not None and page >= total_pages:
            break
        page += 1
    return out

print("Helpers ready. Exports →", OUT_DIR.resolve())

## 3. List all datasets

Browse what's on the project, then copy a name into the next cell.

In [ ]:
datasets = _paginate(langfuse.api.datasets.list)

ds_rows = [{
    "name": d.name,
    "id": getattr(d, "id", None),
    "description": getattr(d, "description", None),
    "items?": "—",
    "created_at": _flatten(getattr(d, "created_at", None)),
    "updated_at": _flatten(getattr(d, "updated_at", None)),
} for d in datasets]

df_datasets = pd.DataFrame(ds_rows)
print(f"{len(df_datasets)} dataset(s) found\n")
df_datasets

## 4. Pick a dataset

Set `DATASET_NAME`. Everything below operates on this dataset.

In [ ]:
DATASET_NAME = "your-dataset-name"   # <-- edit me

# get_dataset hydrates items; api.datasets.get returns metadata
dataset = langfuse.get_dataset(DATASET_NAME)
print(f"Dataset: {DATASET_NAME}")
print(f"Items:   {len(dataset.items)}")

## 5. Mode A — Dataset items only

Just the inputs / expected outputs / metadata. No runs, no traces.

In [ ]:
item_rows = []
for it in dataset.items:
    item_rows.append({
        "item_id": getattr(it, "id", None),
        "status": getattr(it, "status", None),
        "input": _flatten(getattr(it, "input", None)),
        "expected_output": _flatten(getattr(it, "expected_output", None)),
        "metadata": _flatten(getattr(it, "metadata", None)),
        "source_trace_id": getattr(it, "source_trace_id", None),
        "source_observation_id": getattr(it, "source_observation_id", None),
        "created_at": _flatten(getattr(it, "created_at", None)),
    })

df_items = pd.DataFrame(item_rows)
print(f"{len(df_items)} item(s)")
df_items.head(20)

## 6. List runs for this dataset

In v3 "dataset run" == "experiment". We enumerate runs, then drill into run items.

In [ ]:
# api.datasets.get_runs returns paginated runs for a dataset name
def list_runs(dataset_name):
    page, runs = 1, []
    while True:
        resp = langfuse.api.datasets.get_runs(dataset_name, page=page, limit=50)
        data = getattr(resp, "data", []) or []
        if not data:
            break
        runs.extend(data)
        meta = getattr(resp, "meta", None)
        tp = getattr(meta, "total_pages", None) if meta else None
        if tp is not None and page >= tp:
            break
        page += 1
    return runs

runs = list_runs(DATASET_NAME)

run_rows = [{
    "run_name": r.name,
    "run_id": getattr(r, "id", None),
    "description": getattr(r, "description", None),
    "metadata": _flatten(getattr(r, "metadata", None)),
    "created_at": _flatten(getattr(r, "created_at", None)),
} for r in runs]

df_runs = pd.DataFrame(run_rows)
print(f"{len(df_runs)} run(s) for '{DATASET_NAME}'")
df_runs

## 7. Mode B — Runs + outputs + trace reasoning

Pick which runs to pull. For each run item we fetch the linked **trace** and its
**observations** (generations) so you get the model's actual output and intermediate reasoning.

- `SELECTED_RUNS = []`  → all runs
- `SELECTED_RUNS = ["run-a", "run-b"]` → only those

`INCLUDE_OBSERVATIONS = True` pulls every observation per trace (reasoning steps). Set
`False` for a lighter pull (trace-level input/output only).

In [ ]:
SELECTED_RUNS = []              # [] = all runs, or e.g. ["baseline-v1", "gpt4o-run"]
INCLUDE_OBSERVATIONS = True     # fetch per-observation reasoning (slower, richer)
OBS_TEXT_LIMIT = 8000           # truncate long fields for Excel safety (None = no limit)

target_runs = runs if not SELECTED_RUNS else [r for r in runs if r.name in SELECTED_RUNS]
print(f"Pulling {len(target_runs)} run(s)...")

def fetch_run_items(run):
    items = []
    page = 1
    while True:
        resp = langfuse.api.dataset_run_items.list(
            dataset_id=run.dataset_id,
            run_name=run.name,
            page=page, limit=100,
        )
        data = getattr(resp, "data", []) or []
        if not data:
            break
        items.extend(data)
        page += 1
    return items

# trace + observation cache to avoid refetching
_trace_cache, _obs_cache = {}, {}

def get_trace(trace_id):
    if not trace_id:
        return None
    if trace_id not in _trace_cache:
        try:
            _trace_cache[trace_id] = langfuse.api.trace.get(trace_id)
        except Exception as e:
            _trace_cache[trace_id] = None
            print(f"  ! trace {trace_id} fetch failed: {e}")
    return _trace_cache[trace_id]

def get_observations(trace_id):
    if not trace_id:
        return []
    if trace_id not in _obs_cache:
        obs = []
        page = 1
        try:
            while True:
                resp = langfuse.api.observations.get_many(
                    trace_id=trace_id, page=page, limit=100
                )
                data = getattr(resp, "data", []) or []
                if not data:
                    break
                obs.extend(data)
                page += 1
        except Exception as e:
            print(f"  ! observations {trace_id} failed: {e}")
        _obs_cache[trace_id] = obs
    return _obs_cache[trace_id]

flat_rows = []      # one row per run item (for Excel)
nested = []         # full structured tree (for JSON)

for run in target_runs:
    ri = fetch_run_items(run)
    print(f"  run '{run.name}': {len(ri)} item(s)")
    run_node = {"run": _to_jsonable(run), "items": []}

    for it in ri:
        trace_id = getattr(it, "trace_id", None)
        item_id  = getattr(it, "dataset_item_id", None)

        # expected output from the dataset item
        expected = None
        try:
            di = langfuse.api.dataset_items.get(item_id) if item_id else None
            expected = getattr(di, "expected_output", None) if di else None
        except Exception:
            di = None

        trace = get_trace(trace_id)
        observations = get_observations(trace_id) if INCLUDE_OBSERVATIONS else []

        # reasoning = concatenated generation outputs / observation chain
        reasoning_parts = []
        for o in observations:
            o_type = getattr(o, "type", "")
            name   = getattr(o, "name", "")
            o_out  = getattr(o, "output", None)
            if o_out is not None:
                reasoning_parts.append(f"[{o_type}:{name}] {_flatten(o_out, 2000)}")
        reasoning = "\n".join(reasoning_parts) if reasoning_parts else None

        flat_rows.append({
            "run_name": run.name,
            "run_id": getattr(run, "id", None),
            "dataset_item_id": item_id,
            "run_item_id": getattr(it, "id", None),
            "trace_id": trace_id,
            "observation_id": getattr(it, "observation_id", None),
            "input": _flatten(getattr(trace, "input", None) if trace else None, OBS_TEXT_LIMIT),
            "actual_output": _flatten(getattr(trace, "output", None) if trace else None, OBS_TEXT_LIMIT),
            "expected_output": _flatten(expected, OBS_TEXT_LIMIT),
            "reasoning_chain": _flatten(reasoning, OBS_TEXT_LIMIT),
            "n_observations": len(observations),
            "trace_name": getattr(trace, "name", None) if trace else None,
            "latency_ms": getattr(trace, "latency", None) if trace else None,
            "total_cost": getattr(trace, "total_cost", None) if trace else None,
        })

        run_node["items"].append({
            "run_item": _to_jsonable(it),
            "dataset_item": _to_jsonable(di),
            "trace": _to_jsonable(trace),
            "observations": _to_jsonable(observations),
        })

    nested.append(run_node)

df_run_items = pd.DataFrame(flat_rows)
print(f"\nTotal run items pulled: {len(df_run_items)}")
df_run_items.head(20)

## 8. Quick inspection

Look at one full record (trace + reasoning) before exporting.

In [ ]:
if not df_run_items.empty:
    row = df_run_items.iloc[0]
    print("RUN      :", row["run_name"])
    print("TRACE    :", row["trace_id"])
    print("\nINPUT\n", str(row["input"])[:1500])
    print("\nEXPECTED\n", str(row["expected_output"])[:1500])
    print("\nACTUAL\n", str(row["actual_output"])[:1500])
    print("\nREASONING CHAIN\n", str(row["reasoning_chain"])[:3000])
else:
    print("No run items — run Mode B first.")

## 9. Export to Excel + JSON

Choose what to write. Each export is timestamped under `langfuse_exports/`.

- **Excel**: one workbook, sheets = `datasets`, `items`, `runs`, `run_items`
- **JSON**: flat run-items file *plus* the full nested tree (traces + observations)

In [ ]:
EXPORT_DATASETS   = True
EXPORT_ITEMS      = True
EXPORT_RUNS       = True
EXPORT_RUN_ITEMS  = True
EXPORT_NESTED_JSON = True   # full trace/observation tree

stamp = _ts()
safe_name = quote(DATASET_NAME, safe="")

# ---------- Excel ----------
xlsx_path = OUT_DIR / f"{safe_name}_{stamp}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xl:
    wrote = False
    if EXPORT_DATASETS and 'df_datasets' in dir() and not df_datasets.empty:
        df_datasets.to_excel(xl, sheet_name="datasets", index=False); wrote = True
    if EXPORT_ITEMS and 'df_items' in dir() and not df_items.empty:
        df_items.to_excel(xl, sheet_name="items", index=False); wrote = True
    if EXPORT_RUNS and 'df_runs' in dir() and not df_runs.empty:
        df_runs.to_excel(xl, sheet_name="runs", index=False); wrote = True
    if EXPORT_RUN_ITEMS and 'df_run_items' in dir() and not df_run_items.empty:
        df_run_items.to_excel(xl, sheet_name="run_items", index=False); wrote = True
    if not wrote:
        pd.DataFrame({"note": ["nothing selected/empty"]}).to_excel(xl, sheet_name="empty", index=False)
print("Excel  →", xlsx_path)

# ---------- JSON ----------
json_payload = {
    "exported_at": stamp,
    "host": LANGFUSE_HOST,
    "dataset_name": DATASET_NAME,
}
if EXPORT_ITEMS and 'df_items' in dir():
    json_payload["items"] = df_items.to_dict(orient="records")
if EXPORT_RUNS and 'df_runs' in dir():
    json_payload["runs"] = df_runs.to_dict(orient="records")
if EXPORT_RUN_ITEMS and 'df_run_items' in dir():
    json_payload["run_items_flat"] = df_run_items.to_dict(orient="records")

json_path = OUT_DIR / f"{safe_name}_{stamp}.json"
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2))
print("JSON   →", json_path)

if EXPORT_NESTED_JSON and 'nested' in dir() and nested:
    nested_path = OUT_DIR / f"{safe_name}_{stamp}_nested.json"
    nested_path.write_text(json.dumps(_to_jsonable(nested), ensure_ascii=False, indent=2))
    print("Nested →", nested_path)

print("\nDone.")

---
### Notes / gotchas (v3 OSS)

- **`dataset_run_items.list`** needs both `dataset_id` (from the run object) and `run_name`.
- **Reasoning** comes from per-observation `output`s on the trace. If your app doesn't emit
  intermediate generations, you'll only get trace-level `input`/`output`.
- Self-hosted ingestion lags ~15–30s; very recent runs may not appear immediately.
- If `api.datasets.get_runs` isn't present on your exact build, fall back to the REST route
  `GET /api/public/datasets/{name}/runs` via `langfuse.api` or `langfuse.get_run(dataset_name=, run_name=)`.
- Excel cells are capped via `OBS_TEXT_LIMIT`; the full untruncated content is always in the JSON exports.